# Bounded Forex RL pipeline on real EUR/USD data

This notebook fetches a short daily EUR/USD history from Yahoo Finance through the project's `download_eurusd` function, then trains a deliberately small DQN on one walk-forward fold. Its saved outputs are empirical results from that bounded real-data run. They are not evidence of a profitable or deployable strategy. Run all cells after installing `.[research]`.


In [1]:
from dataclasses import replace
from datetime import datetime, timezone
from time import perf_counter
import pandas as pd
import yfinance as yf
from stable_baselines3 import DQN
from forex_rl import ExperimentConfig, download_eurusd, compute_indicators, build_walkforward_folds, make_scaled_env, backtest_model, run_baselines, seed_everything
from forex_rl.walk_forward import slice_fold

started = perf_counter()
config = replace(ExperimentConfig(), symbol='EURUSD=X', start_date='2023-01-01',
                 end_date='2023-08-01', random_seed=7, window_size=4, min_train_months=3,
                 val_months=1, test_months=1, step_months=1, n_envs=1, wf_n_envs=1,
                 total_timesteps=96, eval_freq=0, max_bars_in_trade=3,
                 transaction_cost=0.0001, slippage=0.00005)
seed_everything(config.random_seed)
feature_cols = ['ema20', 'rsi', 'macd', 'macd_signal', 'macd_hist', 'bb_upper', 'bb_middle', 'bb_lower', 'atr_n', 'ret']
price_cols = ['open', 'high', 'low', 'close', 'atr']
print('Source: Yahoo Finance via yfinance', yf.__version__, 'ticker:', config.symbol)
print('Requested dates:', config.start_date, 'through', config.end_date, '(end exclusive)')
print('Training steps:', config.total_timesteps, 'DQN learning starts: 8; CPU')
print('Exact experiment config:', config.to_dict())


Source: Yahoo Finance via yfinance 1.7.0 ticker: EURUSD=X
Requested dates: 2023-01-01 through 2023-08-01 (end exclusive)
Training steps: 96 DQN learning starts: 8; CPU
Exact experiment config: {'symbol': 'EURUSD=X', 'start_date': '2023-01-01', 'end_date': '2023-08-01', 'window_size': 4, 'position_risk_frac': 0.01, 'sl_atr_mult': 1.8, 'tp_atr_mult': 2.2, 'max_bars_in_trade': 3, 'transaction_cost': 0.0001, 'slippage': 5e-05, 'invalid_action_penalty': 0.0, 'dd_penalty': 0.0, 'turnover_penalty': 0.0, 'min_train_months': 3, 'val_months': 1, 'test_months': 1, 'step_months': 1, 'use_rolling': True, 'rolling_train_months': 36, 'random_seed': 7, 'algo': 'DQN', 'total_timesteps': 96, 'eval_freq': 0, 'n_envs': 1, 'wf_n_envs': 1, 'rppo_n_steps': 128, 'qrdqn_n_quantiles': 51, 'min_trades_val': 4, 'log_dir': './logs_rl', 'report_dir': './logs_rl/reports'}


In [2]:
retrieved_at = datetime.now(timezone.utc).isoformat()
raw = download_eurusd(config.symbol, config.start_date, config.end_date)
assert 100 <= len(raw) <= 180, 'Expected a bounded real-data sample; inspect source/date range.'
data = compute_indicators(raw)
fold = build_walkforward_folds(data, config.min_train_months, config.val_months,
                               config.test_months, config.step_months)[0]
train, validation, test = (slice_fold(data, fold, part) for part in ('train', 'val', 'test'))
assert train.index.max() < validation.index.min() < test.index.min()
print('Retrieved at UTC:', retrieved_at)
print('Source row count and observed dates:', len(raw), raw.index.min(), raw.index.max())
print('Prepared rows:', len(data))
print('Train/validation/test rows:', len(train), len(validation), len(test))
print('Fold boundaries (end exclusive):', fold)


Retrieved at UTC: 2026-09-29T13:38:17.308383+00:00
Source row count and observed dates: 151 2023-01-02 00:00:00 2023-07-31 00:00:00
Prepared rows: 132
Train/validation/test rows: 64 22 21
Fold boundaries (end exclusive): {'fold': 1, 'train_start': Timestamp('2023-01-27 00:00:00'), 'train_end': Timestamp('2023-04-27 00:00:00'), 'val_start': Timestamp('2023-04-27 00:00:00'), 'val_end': Timestamp('2023-05-27 00:00:00'), 'test_start': Timestamp('2023-05-27 00:00:00'), 'test_end': Timestamp('2023-06-27 00:00:00')}


In [3]:
train_env, validation_env, scaler = make_scaled_env(
    train, validation, feature_cols, price_cols, config.environment_kwargs,
    config.random_seed, n_envs=1)
assert scaler.means.equals(train[feature_cols].mean())
model = DQN('MlpPolicy', train_env, seed=config.random_seed, device='cpu', verbose=0,
            policy_kwargs={'net_arch': [32, 32]}, learning_rate=0.001,
            buffer_size=256, learning_starts=8, batch_size=8, train_freq=1,
            gradient_steps=1)
try:
    model.learn(total_timesteps=config.total_timesteps)
finally:
    train_env.close()
    validation_env.close()
print('Actual model timesteps:', model.num_timesteps)


Actual model timesteps: 96


In [4]:
validation_result = backtest_model(model, scaler, validation, feature_cols, price_cols, config)
agent = backtest_model(model, scaler, test, feature_cols, price_cols, config)
baselines = run_baselines(test, config)
rows = {'DQN test': agent, **{name + ' test': value for name, value in baselines.items()}}
summary = pd.DataFrame({name: {'trades': result['metrics']['trade_count'],
                               'total_R': result['metrics']['total_R'],
                               'sharpe': result['metrics']['sharpe'],
                               'sortino': result['metrics']['sortino'],
                               'maxDD_pct': result['metrics']['maxDD_pct'],
                               'turnover': result['metrics']['turnover']}
                        for name, result in rows.items()}).T
print('Validation (model selection was not performed):', validation_result['metrics'])
print('Held-out test and matched baselines:')
print(summary.to_string())
print('Elapsed seconds:', round(perf_counter() - started, 2))


Validation (model selection was not performed): {'trades': 2, 'win_rate': 0.0, 'profit_factor': 0.0, 'total_R': -0.5362968581551679, 'avg_R': -0.26814842907758396, 'maxDD_R': -0.5362968581551679, 'MAR_R': -1.0, 'sharpe': -3.0768187286520496, 'sortino': -3.6571687212859487, 'maxDD_pct': -0.005356272101418202, 'CAGR': -0.059664511575027834, 'calmar': -11.139186069212244, 'turnover': 2.944422427882161, 'trade_count': 2}
Held-out test and matched baselines:
             trades   total_R    sharpe   sortino  maxDD_pct  turnover
DQN test        5.0  1.073788  2.744719  5.768991  -0.004902  7.706522
flat test       0.0  0.000000       NaN       NaN   0.000000  0.000000
random test     0.0  0.000000       NaN       NaN   0.000000  0.000000
trend test      5.0  1.182249  3.032024  6.015724  -0.005641  7.725731
Elapsed seconds: 4.03


## Interpretation

These outputs use a real but short EUR/USD daily sample, one walk-forward fold, and 96 DQN training steps. The run checks the data, training, and evaluation path. It does not establish statistical significance, production execution realism, stability across market regimes, or profitability. No full historical walk-forward run, large-model training, model-selection search, or transaction-cost calibration was performed. Sharpe/Sortino can be undefined on small samples with little return variation.
